In [4]:
import os
import json
import torch
import clip
import faiss
import numpy as np
import cv2
from PIL import Image
from tqdm import tqdm
from scenedetect import VideoManager, SceneManager
from scenedetect.detectors import ContentDetector

VIDEO_DIR = r"E:\FPTU\Spring26\DAT301m - SLP301\Dataset_Visual_Extraction\Data\Video\video"
KEYFRAME_OUTPUT_DIR = r"E:\FPTU\Spring26\DAT301m - SLP301\Dataset_Visual_Extraction\Keyframe\keyframes"
BIN_OUTPUT = "dict/faiss_clip_cosine.bin"
JSON_OUTPUT = "dict/keyframes_id.json"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if not os.path.exists(KEYFRAME_OUTPUT_DIR):
    os.makedirs(KEYFRAME_OUTPUT_DIR)
if not os.path.exists("dict"):
    os.makedirs("dict")

d:\Anaconda\envs\muvi-search\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
def extract_smart_keyframes(video_path, output_folder):
    filename = os.path.basename(video_path).split('.')[0]
    video_subfolder = os.path.join(output_folder, filename)
    if not os.path.exists(video_subfolder):
        os.makedirs(video_subfolder)
    
    video_manager = VideoManager([video_path])
    scene_manager = SceneManager()
    scene_manager.add_detector(ContentDetector(threshold=27.0))
    video_manager.set_downscale_factor()
    
    try:
        video_manager.start()
        scene_manager.detect_scenes(frame_source=video_manager)
        scene_list = scene_manager.get_scene_list(video_manager.get_base_timecode())
        
        cap = cv2.VideoCapture(video_path)
        count = 0
        
        if not scene_list:
             scene_list = [(video_manager.get_base_timecode(), video_manager.get_base_timecode())]

        for scene in scene_list:
            start_frame_idx = scene[0].get_frames()
            cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame_idx)
            ret, frame = cap.read()
            
            if ret:
                out_path = os.path.join(video_subfolder, f"{filename}_f{start_frame_idx}.jpg")
                cv2.imwrite(out_path, frame)
                count += 1
                
        cap.release()
        video_manager.release()
        return count
        
    except Exception as e:
        print(f"Error processing {filename}: {e}")
        return 0

In [7]:
import os
from tqdm import tqdm
LOG_FILE = "dict/processed_videos_log.txt"
processed_videos = set()
if os.path.exists(LOG_FILE):
    with open(LOG_FILE, "r", encoding="utf-8") as f:
        processed_videos = set(line.strip() for line in f if line.strip())

print(f"Đã tìm thấy {len(processed_videos)} video đã xử lý trước đó.")

video_files = [f for f in os.listdir(VIDEO_DIR) if f.endswith(('.mp4', '.mkv', '.avi'))]
print(f"Tổng cộng {len(video_files)} video trong thư mục nguồn.")
total_extracted = 0

for vid in tqdm(video_files, desc="Smart Extraction"):
    if vid in processed_videos:
        continue
    
    full_path = os.path.join(VIDEO_DIR, vid)
    num = extract_smart_keyframes(full_path, KEYFRAME_OUTPUT_DIR)
    total_extracted += num
    
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(vid + "\n")

print(f"\n Đã trích xuất thêm được {total_extracted} keyframes.")

Đã tìm thấy 146 video đã xử lý trước đó.
Tổng cộng 216 video trong thư mục nguồn.


Smart Extraction:   0%|          | 0/216 [00:00<?, ?it/s]VideoManager is deprecated and will be removed.
`base_timecode` argument is deprecated and has no effect.
Smart Extraction:  68%|██████▊   | 147/216 [00:50<00:23,  2.94it/s]VideoManager is deprecated and will be removed.
`base_timecode` argument is deprecated and has no effect.
Smart Extraction:  69%|██████▊   | 148/216 [01:51<01:03,  1.07it/s]VideoManager is deprecated and will be removed.
`base_timecode` argument is deprecated and has no effect.
Smart Extraction:  69%|██████▉   | 149/216 [03:06<02:09,  1.94s/it]VideoManager is deprecated and will be removed.
`base_timecode` argument is deprecated and has no effect.
Smart Extraction:  69%|██████▉   | 150/216 [04:52<04:18,  3.91s/it]VideoManager is deprecated and will be removed.
`base_timecode` argument is deprecated and has no effect.
Smart Extraction:  70%|██████▉   | 151/216 [06:08<06:17,  5.80s/it]VideoManager is deprecated and will be removed.
`base_timecode` argument is de


 Đã trích xuất thêm được 7450 keyframes.


In [12]:
print(f"Loading CLIP (ViT-B/16) on {DEVICE}...")

model, preprocess = clip.load("ViT-B/16", device=DEVICE, jit=False)

Loading CLIP (ViT-B/16) on cuda...


In [15]:
from torch.utils.data import Dataset, DataLoader


BATCH_SIZE = 256


NUM_WORKERS = 0 

class KeyframeDataset(Dataset):
    def __init__(self, image_paths, preprocess):
        self.image_paths = image_paths
        self.preprocess = preprocess

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        try:
            img = Image.open(path)
            image_tensor = self.preprocess(img)
            return image_tensor, path
        except Exception as e:
            return torch.zeros(3, 224, 224), "ERROR"

import glob

search_pattern = os.path.join(KEYFRAME_OUTPUT_DIR, "**", "*.jpg")
image_paths = sorted(glob.glob(search_pattern, recursive=True))
print(f"Found {len(image_paths)} frames.")

dataset = KeyframeDataset(image_paths, preprocess)
dataloader = DataLoader(
    dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS, 
    pin_memory=True 
)

print(f"🚀 Starting embedding with Batch Size {BATCH_SIZE}...")

features_list = []
id2path = {}
global_idx = 0

with torch.no_grad():
    for batch_images, batch_paths in tqdm(dataloader):
        valid_mask = [p != "ERROR" for p in batch_paths]
        if not any(valid_mask): continue
        
        real_images = batch_images[valid_mask].to(DEVICE)
        
        batch_features = model.encode_image(real_images)
        batch_features /= batch_features.norm(dim=-1, keepdim=True)
        
        features_list.append(batch_features.cpu().numpy())
        
        for path in batch_paths:
            if path == "ERROR": continue
            
            clean_path = path.replace("\\", "/")
            try:
                rel_path = os.path.relpath(clean_path, start=KEYFRAME_OUTPUT_DIR.replace("\\", "/"))
                id2path[global_idx] = rel_path
            except:
                id2path[global_idx] = clean_path
            global_idx += 1

Found 32311 frames.
🚀 Starting embedding with Batch Size 256...


100%|██████████| 127/127 [14:40<00:00,  6.93s/it]


In [16]:
if features_list:
    print("\nSaving FAISS Index...")
    
    all_features = np.concatenate(features_list)
    all_features = all_features.astype('float32')
    
    d = all_features.shape[1] 
    index = faiss.IndexFlatIP(d)
    index.add(all_features)
    
    faiss.write_index(index, BIN_OUTPUT)
    
    with open(JSON_OUTPUT, 'w', encoding='utf-8') as f:
        json.dump(id2path, f, indent=4) 
        
    print(f" Đã lưu Index ({all_features.shape[0]} vectors) và Mapping.")
else:
    print("Không có vector nào được trích xuất. Hãy kiểm tra lại Cell 5.")


Saving FAISS Index...
 Đã lưu Index (32311 vectors) và Mapping.


SigLIP

In [17]:
import os
import glob
import json
import torch
import numpy as np
import faiss
import open_clip
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

VIDEO_DIR = r"E:\FPTU\Spring26\DAT301m - SLP301\Dataset_Visual_Extraction\Data\Video\video"
KEYFRAME_OUTPUT_DIR = r"E:\FPTU\Spring26\DAT301m - SLP301\Dataset_Visual_Extraction\Keyframe\keyframes"

BIN_OUTPUT_SIGLIP = "dict/faiss_siglip.bin"
JSON_OUTPUT_SIGLIP = "dict/keyframes_id_siglip.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if not os.path.exists("dict"):
    os.makedirs("dict")

In [18]:
model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-16-SigLIP', 
    pretrained='webli', 
    device=DEVICE
)
model.eval()
print(f"Loaded SigLIP on {DEVICE}")

d:\Anaconda\envs\muvi-search\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ACER\.cache\huggingface\hub\models--timm--ViT-B-16-SigLIP. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loaded SigLIP on cuda


In [19]:
BATCH_SIZE = 128
NUM_WORKERS = 0

class KeyframeDataset(Dataset):
    def __init__(self, image_paths, preprocess):
        self.image_paths = image_paths
        self.preprocess = preprocess

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        try:
            img = Image.open(path)
            image_tensor = self.preprocess(img)
            return image_tensor, path
        except:
            return torch.zeros(3, 224, 224), "ERROR"

search_pattern = os.path.join(KEYFRAME_OUTPUT_DIR, "**", "*.jpg")
image_paths = sorted(glob.glob(search_pattern, recursive=True))
print(f"Found {len(image_paths)} images")

dataset = KeyframeDataset(image_paths, preprocess)
dataloader = DataLoader(
    dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS, 
    pin_memory=True
)

features_list = []
id2path = {}
global_idx = 0

print("Starting embedding process...")

with torch.no_grad():
    for batch_images, batch_paths in tqdm(dataloader):
        valid_mask = [p != "ERROR" for p in batch_paths]
        if not any(valid_mask):
            continue
        
        real_images = batch_images[valid_mask].to(DEVICE)
        
        batch_features = model.encode_image(real_images)
        batch_features /= batch_features.norm(dim=-1, keepdim=True)
        
        features_list.append(batch_features.cpu().numpy())
        
        for path in batch_paths:
            if path == "ERROR":
                continue
            
            clean_path = path.replace("\\", "/")
            try:
                rel_path = os.path.relpath(clean_path, start=KEYFRAME_OUTPUT_DIR.replace("\\", "/"))
                id2path[global_idx] = rel_path
            except:
                id2path[global_idx] = clean_path
            global_idx += 1

Found 32311 images
Starting embedding process...


100%|██████████| 253/253 [11:14<00:00,  2.67s/it]


In [20]:
if features_list:
    all_features = np.concatenate(features_list)
    all_features = all_features.astype('float32')
    
    d = all_features.shape[1]
    index = faiss.IndexFlatIP(d)
    index.add(all_features)
    
    faiss.write_index(index, BIN_OUTPUT_SIGLIP)
    
    with open(JSON_OUTPUT_SIGLIP, 'w', encoding='utf-8') as f:
        json.dump(id2path, f, indent=4)
        
    print(f"Saved SigLIP index to {BIN_OUTPUT_SIGLIP}")
    print(f"Saved ID mapping to {JSON_OUTPUT_SIGLIP}")
else:
    print("No features extracted")

Saved SigLIP index to dict/faiss_siglip.bin
Saved ID mapping to dict/keyframes_id_siglip.json
